# 01 · Python Preliminaries for Coding Interviews
**Squarepoint SWE Intern prep, first round Oct 9**

This notebook is a warm-up for someone who knows programming (C/C++) but never learned Python from scratch.
Run every cell top to bottom (`Shift+Enter`). Change things and re-run them: that's how it sticks.

| Part | What | Time |
|---|---|---|
| 1 | Python mental model (vs. C++) | 15 min |
| 2 | Syntax essentials: control flow, functions | 25 min |
| 3 | Strings | 20 min |
| 4 | Lists, tuples, unpacking | 25 min |
| 5 | Dicts, sets, `collections` | 30 min |
| 6 | Interview toolbox: `deque`, `heapq`, `bisect`, `itertools`, `functools` | 30 min |
| 7 | Classes & dataclasses | 15 min |
| 8 | Files, JSON, regex, exceptions | 25 min |
| 9 | Iterators, generators, decorators, context managers | 25 min |
| 10 | Pitfalls that bite in interviews | 15 min |
| 11 | Complexity cheat sheet + mini drills | 30 min |

Parts 1–6 are the must-haves for live coding. Parts 8–9 double as trivia prep (notebook 03 goes deeper).

---
## Part 1 · The Python mental model (coming from C++)

| Idea | C++ | Python |
|---|---|---|
| Typing | static, declared | dynamic: *names* have no type, *objects* do |
| Variables | a box holding a value | a **name bound to an object** (like a reference that can be re-pointed) |
| Blocks | `{ }` | indentation (4 spaces) |
| Memory | manual / RAII | automatic: reference counting + cycle GC |
| Integers | fixed width, can overflow | **arbitrary precision**, never overflow |
| Passing args | by value / by reference | "by object reference": the function gets a new name for the same object |
| `main` | required | none; the file runs top to bottom |
| Compile | ahead of time | compiled to bytecode, run by an interpreter (CPython) |

The single most important idea: **`=` never copies. It binds a name to an object.**

In [3]:
a = [1, 2, 3]
b = a            # b is another name for the SAME list
b.append(4)
print(a)         # [1, 2, 3, 4]  <- a changed too!
print(a is b)    # True: same object (identity)
print(id(a) == id(b))

c = a.copy()        # a shallow copy (also list(a) or a.copy())
c.append(5)
print(a, c, a is c)

[1, 2, 3, 4]
True
True
[1, 2, 3, 4] [1, 2, 3, 4, 5] False


**Mutable vs immutable.** This decides whether "changing" something modifies it in place or makes a new object.

- Immutable: `int`, `float`, `bool`, `str`, `tuple`, `frozenset`, `None`
- Mutable: `list`, `dict`, `set`, most user classes

Only immutable (more precisely, *hashable*) objects can be dict keys or set members.

In [4]:
s = "abc"
t = s
s += "d"          # creates a NEW string; t still points to the old one
print(s, t)

x = 10
def bump(n):
    n += 1        # rebinds local name n; caller's x unaffected
    return n
print(bump(x), x)

def add_item(lst):
    lst.append(99)  # mutates the shared object; caller sees it
nums = [1]
add_item(nums)
print(nums)

abcd abc
11 10
[1, 99]


In [5]:
# Integers never overflow; floats are IEEE-754 doubles (same as C++ double)
print(2 ** 100)
print(0.1 + 0.2 == 0.3, abs((0.1 + 0.2) - 0.3) < 1e-9)
print(7 // 2, -7 // 2)    # floor division: 3, -4  (C++ truncates: -3!)
print(7 % 3, -7 % 3)      # 1, 2   (sign follows the divisor, unlike C++)
print(int(-7 / 2))        # -3: truncation toward zero, if you need C++ behaviour
print(float('inf') > 10**18, float('-inf'))

1267650600228229401496703205376
False True
3 -4
1 2
-3
True -inf


> ⚠️ **Interview trap:** `-7 // 2 == -4` in Python. In calculator problems (LC 227) that do integer division that
> truncates toward zero, use `int(a / b)`, not `a // b`.

**Truthiness.** Empty containers, `0`, `0.0`, `""`, `None` are falsy. Everything else is truthy.

In [6]:
for v in [0, 1, "", "a", [], [0], {}, None]:
    print(repr(v), "->", bool(v))

stack = []
if not stack:          # idiomatic empty check (not `len(stack) == 0`)
    print("empty")

x = None
print(x is None)       # always compare to None with `is`

0 -> False
1 -> True
'' -> False
'a' -> True
[] -> False
[0] -> True
{} -> False
None -> False
empty
True


---
## Part 2 · Syntax essentials
### Control flow

In [7]:
n = 7
if n < 0:
    print("neg")
elif n == 0:
    print("zero")
else:
    print("pos")

label = "even" if n % 2 == 0 else "odd"     # ternary (C++: cond ? a : b)
print(label)

print(0 < n < 10)     # chained comparison, reads like maths
print(n > 5 and n < 9, not n, n == 7 or n == 8)   # and / or / not, not && || !

pos
odd
True
True False True


In [8]:
# for loops iterate over ITEMS, not indices
arr = ["a", "b", "c"]
for x in arr:
    print(x, end=" ")
print()

for i in range(len(arr)):          # indices (range(start, stop, step), stop is exclusive)
    print(i, end=" ")
print()

for i, x in enumerate(arr):        # index + item: use this instead of range(len())
    print(i, x, end=" | ")
print()

for x, y in zip([1, 2, 3], "xyz"): # parallel iteration, stops at shortest
    print(x, y, end=" | ")
print()

for i in range(10, 0, -3):         # counting down
    print(i, end=" ")
print()

for i in reversed(range(3)):
    print(i, end=" ")

a b c 
0 1 2 
0 a | 1 b | 2 c | 
1 x | 2 y | 3 z | 
10 7 4 1 
2 1 0 

In [9]:
# while, break, continue, and the rarely-known for...else
i = 0
while i < 5:
    i += 1          # no i++ in Python!
    if i == 2:
        continue
    if i == 4:
        break
    print(i)

for x in [3, 5, 7]:
    if x % 2 == 0:
        print("found even")
        break
else:               # runs only if the loop did NOT break
    print("no even number")

1
3
no even number


### Comprehensions: the most Pythonic thing you will write

In [10]:
squares = [x * x for x in range(6)]
evens   = [x for x in range(10) if x % 2 == 0]
pairs   = [(i, j) for i in range(3) for j in range(i)]     # nested: outer loop first
grid    = [[0] * 3 for _ in range(2)]                       # 2x3 matrix (correct way!)
lengths = {w: len(w) for w in ["hi", "there"]}              # dict comprehension
uniq    = {c for c in "banana"}                              # set comprehension
gen     = (x * x for x in range(10**9))                      # generator: lazy, O(1) memory
print(squares, evens, pairs, grid, lengths, uniq, next(gen), sep="\n")
print(sum(x * x for x in range(4)))   # generator directly inside a call

[0, 1, 4, 9, 16, 25]
[0, 2, 4, 6, 8]
[(1, 0), (2, 0), (2, 1)]
[[0, 0, 0], [0, 0, 0]]
{'hi': 2, 'there': 5}
{'a', 'b', 'n'}
0
14


### Functions

In [11]:
def area(w, h=1):                  # default argument
    """Docstring: describe what the function does."""
    return w * h

print(area(3), area(3, 4), area(h=5, w=2))     # keyword arguments

def stats(*nums, **opts):          # *args -> tuple, **kwargs -> dict
    print("nums =", nums, "opts =", opts)
stats(1, 2, 3, precision=2)

def min_max(xs):
    return min(xs), max(xs)        # returning multiple values = returning a tuple
lo, hi = min_max([4, 1, 9])
print(lo, hi)

square = lambda x: x * x           # anonymous one-liner; mainly used as sort keys
print(square(5))

def typed(xs: list[int], k: int) -> int:   # type hints: documentation only, not enforced
    return xs[k]
print(typed([1, 2], 1))

3 12 10
nums = (1, 2, 3) opts = {'precision': 2}
1 9
25
2


In [12]:
# Closures and nonlocal: inner functions capture outer variables
def make_counter():
    count = 0
    def inc():
        nonlocal count        # without this, `count += 1` would raise UnboundLocalError
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())

# In interview code you'll often write a nested helper for DFS/recursion:
def solve(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()
    def dfs(r, c):                     # reads rows/cols/seen from the enclosing scope
        if not (0 <= r < rows and 0 <= c < cols) or (r, c) in seen:
            return
        seen.add((r, c))               # mutating is fine without nonlocal; rebinding is not
    dfs(0, 0)
    return len(seen)
print(solve([[1]]))

1 2 3
1


In [13]:
# ⚠️ Classic trap: mutable default argument is created ONCE, at definition time
def bad(x, acc=[]):
    acc.append(x)
    return acc
print(bad(1), bad(2))          # [1, 2] [1, 2]  (surprise!)

def good(x, acc=None):
    if acc is None:
        acc = []
    acc.append(x)
    return acc
print(good(1), good(2))

[1, 2] [1, 2]
[1] [2]


---
## Part 3 · Strings
Strings are **immutable** sequences of Unicode characters. There is no `char` type; a character is a string of length 1.

In [32]:
s = "Hello, World"
print(len(s), s[0], s[-1], s[-3:])       # negative indices count from the end
print(s[2:5], s[:5], s[7:], s[::-1])     # slicing [start:stop:step], reversed
print(s.lower(), s.upper(), s.swapcase())
print(s.find("H"), s.find("z"), s.index("W"))   # find -> -1 if missing; index raises
print(s.count("l"), "World" in s, s.startswith("He"), s.endswith("!"))
print(s.replace("l", "L", 2))
print("  pad \n".strip(), "|", "xxhixx".strip("x"), "|", "a".lstrip())

12 H d rld
llo Hello World dlroW ,olleH
hello, world HELLO, WORLD hELLO, wORLD
0 -1 7
3 True True False
HeLLo, World
pad | hi | a


In [33]:
csv = "a, b,,c"
print(csv.split(","))              # ['a', ' b', '', 'c']
print("a  b\t c".split())         # no arg: splits on ANY whitespace, drops empties
print("k=v=w".split("=", 1))       # maxsplit
print("-".join(["2026", "10", "09"]))
print(" ".join(map(str, [1, 2, 3])))   # join needs strings

['a', ' b', '', 'c']
['a', 'b', 'c']
['k', 'v=w']
2026-10-09
1 2 3


In [16]:
# Character tests & conversions (heavily used in parsing problems)
for ch in "a7 Z_":
    print(repr(ch), ch.isdigit(), ch.isalpha(), ch.isalnum(), ch.isspace(), ch.isupper())

print(ord("a"), chr(98), ord("c") - ord("a"))    # char <-> code point
print(int("42"), int("-7"), int("ff", 16), float("3.5"), str(10))
print(int("0042"), "7".isdigit(), "-7".isdigit())   # note: "-7".isdigit() is False

'a' False True True False False
'7' True False True False False
' ' False False False True False
'Z' False True True False True
'_' False False False False False
97 b 2
42 -7 255 3.5 10
42 True False


In [17]:
# f-strings for formatting
name, px = "AAPL", 187.4567
print(f"{name}: {px:.2f}")        # 2 decimal places
print(f"{42:05d} {0.1234:.1%} {1234567:,}")
print(f"{name!r} {px=}")           # repr and self-documenting (debugging!)

AAPL: 187.46
00042 12.3% 1,234,567
'AAPL' px=187.4567


In [18]:
# ⚠️ Building strings in a loop: s += c is O(n) each time -> O(n^2) total worst case.
# Collect pieces in a list, then join once: O(n).
parts = []
for i in range(5):
    parts.append(str(i))
print("".join(parts))

# Strings are immutable, so to "edit" one, convert to a list:
chars = list("hello")
chars[0] = "j"
print("".join(chars))

01234
jello


---
## Part 4 · Lists, tuples, unpacking
A `list` is a dynamic array (like `std::vector<PyObject*>`). Know the costs:

| Operation | Cost |
|---|---|
| `a[i]`, `a[i] = x`, `len(a)` | O(1) |
| `a.append(x)`, `a.pop()` | O(1) amortized |
| `a.pop(0)`, `a.insert(0, x)` | **O(n)** → use `deque` |
| `x in a`, `a.index(x)`, `a.remove(x)` | **O(n)** → use a `set` for membership |
| `a[i:j]` | O(j−i) copy |
| `a.sort()`, `sorted(a)` | O(n log n), stable (Timsort) |

In [ ]:
a = [5, 3, 8]
a.append(1)
a.extend([9, 9])          # add many
a.insert(1, 7)
print(a)
print(a.pop(), a.pop(0), a)   # pop last, pop index 0
a.remove(9)                    # removes first occurrence (ValueError if absent)
print(a, len(a), max(a), min(a), sum(a))
print(a.index(8), a.count(8))
a.reverse(); print(a)
print(a + [0], [0] * 3)

[5, 7, 3, 8, 1, 9, 9]
9 5 [7, 3, 8, 1, 9]
[7, 3, 8, 1] 4 8 1 19
2 1
[1, 8, 3, 7]
[1, 8, 3, 7, 0] [0, 0, 0]


In [20]:
# Sorting: sort() is in place and returns None; sorted() returns a new list
nums = [3, 1, 2]
print(nums.sort(), nums)          # None [1, 2, 3]  <- don't write nums = nums.sort()!
print(sorted([3, 1, 2], reverse=True))

words = ["banana", "Apple", "cherry", "fig"]
print(sorted(words))                              # uppercase sorts first (ASCII)
print(sorted(words, key=str.lower))
print(sorted(words, key=len))
print(sorted(words, key=lambda w: (-len(w), w)))  # tuple key: len desc, then alphabetical

trades = [("MSFT", 300), ("AAPL", 150), ("GOOG", 150)]
print(sorted(trades, key=lambda t: (t[1], t[0])))

None [1, 2, 3]
[3, 2, 1]
['Apple', 'banana', 'cherry', 'fig']
['Apple', 'banana', 'cherry', 'fig']
['fig', 'Apple', 'banana', 'cherry']
['banana', 'cherry', 'Apple', 'fig']
[('AAPL', 150), ('GOOG', 150), ('MSFT', 300)]


In [21]:
# 2-D lists
rows, cols = 3, 4
good = [[0] * cols for _ in range(rows)]
bad = [[0] * cols] * rows          # ⚠️ 3 references to the SAME inner list
bad[0][0] = 1
good[0][0] = 1
print(bad)
print(good)

matrix = [[1, 2, 3], [4, 5, 6]]
print([list(col) for col in zip(*matrix)])   # transpose
print([x for row in matrix for x in row])    # flatten

[[1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]]
[[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
[[1, 4], [2, 5], [3, 6]]
[1, 2, 3, 4, 5, 6]


In [22]:
# Tuples: immutable, hashable (usable as dict keys / set items), lightweight records
p = (3, 4)
x, y = p                    # unpacking
print(x, y, p[0], len(p))
single = (5,)               # one-element tuple needs the comma
print(type(single), type((5)))

a, b = 1, 2
a, b = b, a                 # swap without temp
print(a, b)

first, *middle, last = [1, 2, 3, 4, 5]    # star unpacking
print(first, middle, last)

visited = {(0, 0), (1, 2)}  # tuples as coordinates in a set: extremely common in grid problems
print((1, 2) in visited)

from collections import namedtuple
Order = namedtuple("Order", ["id", "side", "px", "qty"])
o = Order(1, "B", 101.5, 100)
print(o.px, o.qty, o)

3 4 3 2
<class 'tuple'> <class 'int'>
2 1
1 [2, 3, 4] 5
True
101.5 100 Order(id=1, side='B', px=101.5, qty=100)


---
## Part 5 · Dicts, sets, and `collections`
`dict` = hash map (like `std::unordered_map`), average O(1) get/set/delete/`in`.
Since Python 3.7, dicts **remember insertion order**.

In [23]:
d = {"AAPL": 150, "MSFT": 300}
d["GOOG"] = 140              # insert / overwrite
print(d["AAPL"], len(d), "MSFT" in d)
print(d.get("TSLA"), d.get("TSLA", 0))    # d["TSLA"] would raise KeyError
del d["MSFT"]
print(d.pop("GOOG"), d)
d.update({"X": 1, "Y": 2})
for k in d:                  # iterates keys
    print(k, end=" ")
print()
for k, v in d.items():       # keys + values: use this most of the time
    print(k, v, end=" | ")
print()
print(list(d.keys()), list(d.values()))

150 3 True
None 0
140 {'AAPL': 150}
AAPL X Y 
AAPL 150 | X 1 | Y 2 | 
['AAPL', 'X', 'Y'] [150, 1, 2]


In [24]:
# Counting and grouping, three ways. Learn the last two.
words = "the cat and the hat and the bat".split()

counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1
print(counts)

from collections import defaultdict, Counter
by_len = defaultdict(list)           # missing key -> default list() created automatically
for w in words:
    by_len[len(w)].append(w)
print(dict(by_len))

c = Counter(words)
print(c, c["the"], c["zzz"])         # missing key -> 0, no error
print(c.most_common(2))
print(Counter("aab") == Counter("aba"))   # anagram check
print(Counter("aabbb") - Counter("ab"))    # multiset difference

{'the': 3, 'cat': 1, 'and': 2, 'hat': 1, 'bat': 1}
{3: ['the', 'cat', 'and', 'the', 'hat', 'and', 'the', 'bat']}
Counter({'the': 3, 'and': 2, 'cat': 1, 'hat': 1, 'bat': 1}) 3 0
[('the', 3), ('and', 2)]
True
Counter({'b': 2, 'a': 1})


In [25]:
# Nested defaultdict: e.g. volume[symbol][side]
from collections import defaultdict
volume = defaultdict(lambda: defaultdict(int))
for sym, side, qty in [("AAPL", "B", 100), ("AAPL", "S", 50), ("MSFT", "B", 10), ("AAPL", "B", 5)]:
    volume[sym][side] += qty
print({k: dict(v) for k, v in volume.items()})

# setdefault: the plain-dict alternative
g = {}
g.setdefault("a", []).append(1)
print(g)

# Merging and inverting
a, b = {"x": 1}, {"y": 2}
print({**a, **b}, a | b)             # a | b needs Python 3.9+
print({v: k for k, v in {"a": 1, "b": 2}.items()})

{'AAPL': {'B': 105, 'S': 50}, 'MSFT': {'B': 10}}
{'a': [1]}
{'x': 1, 'y': 2} {'x': 1, 'y': 2}
{1: 'a', 2: 'b'}


In [26]:
# Sets: hash set, O(1) membership. Great for "seen" tracking and dedup.
s = {1, 2, 3}
s.add(4); s.discard(10); s.remove(1)     # remove raises KeyError if missing, discard doesn't
print(s, 2 in s)
a, b = {1, 2, 3}, {2, 3, 4}
print(a & b, a | b, a - b, a ^ b)         # intersection, union, difference, symmetric diff
print(set([3, 1, 3, 2]))                  # dedup (order NOT preserved)
print(list(dict.fromkeys([3, 1, 3, 2])))  # dedup preserving order
empty = set()                              # {} is an empty DICT, not a set!
print(type({}), type(empty))

{2, 3, 4} True
{2, 3} {1, 2, 3, 4} {1} {1, 4}
{1, 2, 3}
[3, 1, 2]
<class 'dict'> <class 'set'>


---
## Part 6 · The interview toolbox
These modules are the Python equivalent of the STL. Every one of them appears in LC solutions.

| Need | C++ | Python |
|---|---|---|
| stack | `std::stack` | `list` (`append` / `pop` / `[-1]`) |
| queue / deque | `std::queue`, `std::deque` | `collections.deque` |
| priority queue | `std::priority_queue` (max-heap) | `heapq` on a list (**min-heap**) |
| ordered map / `lower_bound` | `std::map`, `lower_bound` | `bisect` on a sorted list (or `sortedcontainers`, if allowed) |
| hash map / set | `unordered_map` / `unordered_set` | `dict` / `set` |
| memoization | manual | `functools.cache` |

In [ ]:
from collections import deque
q = deque([1, 2])
q.append(3)          # right
q.appendleft(0)      # left
print(q, q.popleft(), q.pop(), q, q[0], q[-1])
window = deque(maxlen=3)     # bounded: old items fall off automatically (rolling windows!)
for x in range(6):
    window.append(x)
print(window)

# BFS template
def bfs(start, neighbors):
    seen = {start}
    q = deque([start])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for nxt in neighbors(node):
            if nxt not in seen:
                seen.add(nxt)
                q.append(nxt)
    return order
print(bfs(1, lambda n: [n * 2, n * 2 + 1] if n < 4 else []))

In [ ]:
import heapq
h = []
for x in [5, 1, 8, 3]:
    heapq.heappush(h, x)
print(h[0], heapq.heappop(h), h)       # smallest at h[0]

nums = [5, 1, 8, 3, 9]
heapq.heapify(nums)                    # O(n) in place
print(heapq.nsmallest(2, [5, 1, 8]), heapq.nlargest(2, [5, 1, 8]))

# Max-heap: push negatives
mh = []
for x in [5, 1, 8]:
    heapq.heappush(mh, -x)
print(-heapq.heappop(mh))

# Heap of tuples compares element by element: (priority, tiebreak, payload)
tasks = []
heapq.heappush(tasks, (2, 0, "write tests"))
heapq.heappush(tasks, (1, 1, "fix bug"))
print(heapq.heappop(tasks))

# Keep top-k largest in a size-k MIN-heap: O(n log k)
def top_k(xs, k):
    h = []
    for x in xs:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)
    return sorted(h, reverse=True)
print(top_k([4, 1, 7, 3, 9, 2], 3))

In [ ]:
import bisect
prices = [100, 101, 103, 103, 107]          # must be sorted
print(bisect.bisect_left(prices, 103))      # 2: first index where 103 could go (== lower_bound)
print(bisect.bisect_right(prices, 103))     # 4: after the last 103 (== upper_bound)
print(bisect.bisect_left(prices, 102))      # 2
bisect.insort(prices, 102)                  # insert keeping sorted: O(n) because of shifting
print(prices)

# Hand-rolled binary search: you may be asked to write it
def lower_bound(a, target):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo
print(lower_bound([1, 3, 3, 5], 3), lower_bound([1, 3, 3, 5], 6))

In [ ]:
import itertools as it
print(list(it.accumulate([1, 2, 3, 4])))                 # prefix sums [1, 3, 6, 10]
print(list(it.accumulate([3, 1, 4, 1, 5], max)))         # running max
print(list(it.permutations([1, 2, 3], 2)))
print(list(it.combinations("abc", 2)))
print(list(it.product([0, 1], repeat=2)))
print(list(it.chain([1, 2], [3])))
for key, grp in it.groupby("aaabbc"):                    # groups CONSECUTIVE equal items
    print(key, len(list(grp)), end=" | ")
print()
print(list(it.islice(it.count(10), 3)))
print(list(it.pairwise([1, 4, 9])))                       # Python 3.10+: (1,4), (4,9)

In [ ]:
from functools import cache, lru_cache, reduce, cmp_to_key
import math, operator

@cache                      # memoization in one line; perfect for top-down DP
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(100))

print(reduce(operator.mul, [1, 2, 3, 4]))       # 24
print(math.gcd(12, 18), math.lcm(4, 6), math.isqrt(17), math.inf, math.comb(5, 2))

# Custom comparator (e.g. LC 179 Largest Number)
def cmp(a, b):
    return -1 if a + b > b + a else (1 if a + b < b + a else 0)
print("".join(sorted(["3", "30", "34", "5", "9"], key=cmp_to_key(cmp))))

In [ ]:
# Built-ins you should have at your fingertips
xs = [3, -1, 4]
print(abs(-3), divmod(17, 5), pow(2, 10, 1000), round(2.675, 2))
print(any(x < 0 for x in xs), all(x > -5 for x in xs))
print(max(xs, key=abs), min("apple", "fig", key=len))
print(list(map(int, "1 2 3".split())), list(filter(lambda x: x > 0, xs)))
print(sorted(set("mississippi")), list(reversed(xs)))
print(isinstance(3, int), isinstance([], (list, tuple)), type(3.0).__name__)

---
## Part 7 · Classes (enough for LRU Cache / Trie / linked-list problems)
- `self` is the explicit `this`.
- No access modifiers; `_name` means "private by convention".
- "Dunder" methods (`__init__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__hash__`...) are operator overloading.

In [ ]:
class Node:
    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt
    def __repr__(self):
        return f"Node({self.val})"

head = Node(1, Node(2, Node(3)))
cur = head
while cur:
    print(cur.val, end=" -> ")
    cur = cur.next
print("None")

class Order:
    count = 0                          # class attribute (shared, like static)
    def __init__(self, px, qty):
        self.px, self.qty = px, qty    # instance attributes
        Order.count += 1
    def notional(self):
        return self.px * self.qty
    def __lt__(self, other):           # makes Orders sortable / heap-able
        return self.px < other.px
    @staticmethod
    def validate(qty):
        return qty > 0
    @classmethod
    def from_str(cls, s):              # alternative constructor
        px, qty = s.split("@")
        return cls(float(px), int(qty))

o = Order.from_str("101.5@200")
print(o.notional(), Order.count, Order.validate(-1), sorted([Order(3, 1), Order(1, 1)])[0].px)

In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)                 # auto __init__, __repr__, __eq__, and ordering
class Trade:
    ts: int
    sym: str = field(compare=False)
    px: float = field(compare=False)
    qty: int = field(default=0, compare=False)

t1, t2 = Trade(2, "AAPL", 150.0, 10), Trade(1, "MSFT", 300.0)
print(t1, sorted([t1, t2])[0].sym)

class TrieNode:                        # used in word-search / prefix problems
    def __init__(self):
        self.children = {}
        self.end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    def starts_with(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True

tr = Trie(); tr.insert("squarepoint")
print(tr.starts_with("square"), tr.starts_with("quad"))

---
## Part 8 · Files, JSON, regex, exceptions
Glassdoor mentions "clean JSON data with a script" and "regex over 3 pages of raw data", so this part matters.

In [ ]:
# Exceptions
def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        print("caught:", e)
        return None
    finally:
        pass               # always runs (cleanup)

print(safe_div(1, 0))

try:
    int("abc")
except (ValueError, TypeError) as e:
    print(type(e).__name__, e)

def withdraw(balance, amt):
    if amt > balance:
        raise ValueError(f"insufficient funds: {amt} > {balance}")
    return balance - amt
try:
    withdraw(10, 20)
except ValueError as e:
    print(e)

In [ ]:
# Files: ALWAYS use `with` (context manager closes the file even on exceptions)
with open("demo.txt", "w") as f:
    f.write("line one\nline two\n")
    print("third line", file=f)

with open("demo.txt") as f:          # default mode "r"
    for line in f:                   # iterates lazily, line by line (memory-friendly)
        print(line.rstrip("\n"))

with open("demo.txt") as f:
    print(f.read().splitlines())

import os; os.remove("demo.txt")

In [ ]:
import json
raw = '{"sym": "AAPL", "fills": [{"px": 150.1, "qty": 10}, {"px": 150.2, "qty": null}], "live": true}'
obj = json.loads(raw)                 # str -> Python objects (null->None, true->True)
print(obj["fills"][1]["qty"], type(obj))
print(json.dumps(obj, indent=2, sort_keys=True))   # Python -> str

# with open("data.json") as f: obj = json.load(f)       # file versions: load / dump
# with open("out.json", "w") as f: json.dump(obj, f)

# Type checks you need when walking messy JSON
for v in [1, 1.5, "s", None, True, [1], {"a": 1}]:
    kind = ("dict" if isinstance(v, dict) else "list" if isinstance(v, list)
            else "null" if v is None else type(v).__name__)
    print(repr(v), "->", kind)
print(isinstance(True, int))   # ⚠️ True! bool is a subclass of int: check bool first

### Regex cheat sheet (`import re`)

| Pattern | Means | Pattern | Means |
|---|---|---|---|
| `.` | any char except newline | `\d` `\w` `\s` | digit, word char `[A-Za-z0-9_]`, whitespace |
| `*` `+` `?` | 0+, 1+, 0 or 1 | `\D` `\W` `\S` | negations |
| `{3}` `{2,5}` | exact / range count | `[abc]` `[^abc]` `[a-z0-9]` | char class / negated |
| `*?` `+?` | **non-greedy** | `^` `$` | start / end (per line with `re.M`) |
| `(...)` | capture group | `(?P<name>...)` | named group |
| `(?:...)` | non-capturing group | `a\|b` | alternation |
| `\b` | word boundary | `\.` | a literal dot (escape specials) |

Always write patterns as raw strings: `r"\d+"`.

| Function | Returns |
|---|---|
| `re.search(p, s)` | first match anywhere, or `None` |
| `re.match(p, s)` | match only at the **start** |
| `re.fullmatch(p, s)` | whole string must match |
| `re.findall(p, s)` | list of strings (or tuples of groups) |
| `re.finditer(p, s)` | iterator of match objects (use `.group()`, `.groupdict()`) |
| `re.sub(p, repl, s)` | replace |
| `re.split(p, s)` | split on pattern |

In [ ]:
import re
log = """2026-10-09 09:30:01.123 INFO  order_id=1001 sym=AAPL side=BUY  px=187.25 qty=100
2026-10-09 09:30:01.456 WARN  order_id=1002 sym=MSFT side=SELL px=412.10 qty=50 latency_ms=12
2026-10-09 09:30:02.001 ERROR order_id=1003 sym=AAPL side=BUY  msg="rejected: price band"
"""
print(re.findall(r"sym=(\w+)", log))
print(re.findall(r"px=(\d+\.\d+)", log))
print(re.findall(r"(\w+)=([^\s\"]+)", log.splitlines()[0]))      # key=value pairs -> tuples

pat = re.compile(r"^(?P<date>\S+) (?P<time>\S+) (?P<level>[A-Z]+)\s+(?P<rest>.*)$", re.M)
for m in pat.finditer(log):
    print(m.group("level"), "|", m.groupdict()["time"])

print(re.search(r'msg="(.*?)"', log).group(1))       # non-greedy up to the next quote
print(re.sub(r"\s+", " ", "too    many   spaces"))
print(re.split(r"[,;]\s*", "a, b;c ;d"))
print(bool(re.fullmatch(r"[A-Z]{1,5}", "AAPL")), bool(re.fullmatch(r"[A-Z]{1,5}", "aapl")))
print(re.findall(r"\b\w+@\w+\.com\b", "mail a@b.com or c@d.com now"))
print(re.findall(r"-?\d+(?:\.\d+)?", "moved -3.5 then 12 then 0.25"))   # signed decimals

---
## Part 9 · Iterators, generators, decorators, context managers
These show up both in trivia ("what does `yield` do?") and in clean data-processing code.

In [ ]:
# Generator function: `yield` pauses the function and hands out one value at a time
def read_ticks(n):
    for i in range(n):
        print(f"  producing {i}")
        yield i * 10

g = read_ticks(3)
print(type(g))
print(next(g))          # runs until the first yield
for v in g:             # continues from where it paused
    print("got", v)

# Pipelines of generators: each stage lazy, memory O(1) no matter how big the file
def parse(lines):
    for line in lines:
        sym, px = line.split(",")
        yield sym, float(px)
def only(sym, rows):
    return (r for r in rows if r[0] == sym)
data = ["AAPL,1", "MSFT,2", "AAPL,3"]
print(list(only("AAPL", parse(data))))

def flatten(x):          # recursive generator with `yield from`
    for item in x:
        if isinstance(item, list):
            yield from flatten(item)
        else:
            yield item
print(list(flatten([1, [2, [3, [4]], 5]])))

In [ ]:
import time, functools

def timed(fn):                       # a decorator: takes a function, returns a wrapped function
    @functools.wraps(fn)             # keeps fn's name/docstring
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} took {(time.perf_counter() - t0) * 1e3:.2f} ms")
        return result
    return wrapper

@timed                               # same as: slow_sum = timed(slow_sum)
def slow_sum(n):
    return sum(range(n))
print(slow_sum(10**6))

def retry(times):                    # decorator WITH arguments = a function returning a decorator
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*a, **k):
            for attempt in range(times):
                try:
                    return fn(*a, **k)
                except Exception as e:
                    print("attempt", attempt + 1, "failed:", e)
            raise RuntimeError("all retries failed")
        return wrapper
    return deco

calls = {"n": 0}
@retry(3)
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise IOError("network blip")
    return "ok"
print(flaky())

In [ ]:
from contextlib import contextmanager

class Timer:                          # class-based context manager
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.ms = (time.perf_counter() - self.t0) * 1e3
        print(f"block took {self.ms:.2f} ms")
        return False                  # False = don't swallow exceptions

with Timer():
    sum(range(10**5))

@contextmanager                       # generator-based: code before yield = __enter__, after = __exit__
def tag(name):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("b"):
    print("bold text")

---
## Part 10 · Pitfalls that bite in interviews

In [ ]:
# 1. == vs is
a = [1, 2]; b = [1, 2]
print(a == b, a is b)            # value equality vs identity

# 2. shallow vs deep copy
import copy
grid = [[1, 2], [3, 4]]
sh = grid.copy(); dp = copy.deepcopy(grid)
grid[0][0] = 99
print(sh[0][0], dp[0][0])        # 99 1

# 3. modifying a collection while iterating over it
d = {"a": 1, "b": 2}
for k in list(d):                # iterate over a snapshot of the keys
    if d[k] == 1:
        del d[k]
print(d)

# 4. recursion depth (default ~1000). Deep DFS/memo recursion can blow up
import sys
print(sys.getrecursionlimit())
sys.setrecursionlimit(10**5)      # fine for interview code; or convert to iteration

# 5. late-binding closures in loops
fs = [lambda: i for i in range(3)]
print([f() for f in fs])           # [2, 2, 2]!
fs = [lambda i=i: i for i in range(3)]
print([f() for f in fs])           # [0, 1, 2]

# 6. integer division and rounding
print(-7 // 2, int(-7 / 2), round(0.5), round(1.5), round(2.5))   # banker's rounding!

# 7. strings compare lexicographically
print("10" < "9", int("10") < int("9"))

---
## Part 11 · Complexity cheat sheet + mini drills

| Structure | Access | Search | Insert | Delete | Notes |
|---|---|---|---|---|---|
| `list` | O(1) | O(n) | O(1)* end / O(n) front | O(1) end / O(n) middle | *amortized |
| `deque` | O(1) ends, O(n) middle | O(n) | O(1) both ends | O(1) both ends | |
| `dict` / `set` | O(1) avg | O(1) avg | O(1) avg | O(1) avg | O(n) worst-case collisions |
| `heapq` | O(1) min | O(n) | O(log n) | O(log n) pop-min | `heapify` O(n) |
| sorted list + `bisect` | O(1) | O(log n) | O(n) | O(n) | |
| `str` | O(1) | O(n·m) `in` | new object | new object | immutable |

Try each drill in the empty cell, then compare with the collapsed answer.

### Drill 1: Count characters
Given `s = 'abracadabra'`, return a dict of char → count, **without** `Counter`, then with `Counter`. Print the most common char.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
s = 'abracadabra'
d = {}
for ch in s:
    d[ch] = d.get(ch, 0) + 1
print(d)
from collections import Counter
print(Counter(s).most_common(1)[0][0])
```
</details>

### Drill 2: Group by first letter
`words = ['apple','avocado','banana','blueberry','cherry']` → `{'a': [...], 'b': [...], 'c': [...]}` using `defaultdict`.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
from collections import defaultdict
words = ['apple','avocado','banana','blueberry','cherry']
g = defaultdict(list)
for w in words:
    g[w[0]].append(w)
print(dict(g))
```
</details>

### Drill 3: Sort records
Sort `[('AAPL', 3), ('MSFT', 1), ('GOOG', 3), ('AMZN', 2)]` by count **descending**, ties by name ascending.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
rows = [('AAPL', 3), ('MSFT', 1), ('GOOG', 3), ('AMZN', 2)]
print(sorted(rows, key=lambda r: (-r[1], r[0])))
```
</details>

### Drill 4: Tokenize an expression
Turn `'12+ 3*(45 -6)'` into `['12', '+', '3', '*', '(', '45', '-', '6', ')']`. Do it once with a loop, once with `re.findall`.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
import re
s = '12+ 3*(45 -6)'
toks, i = [], 0
while i < len(s):
    ch = s[i]
    if ch.isdigit():
        j = i
        while j < len(s) and s[j].isdigit():
            j += 1
        toks.append(s[i:j]); i = j
        continue
    if ch in '+-*/()':
        toks.append(ch)
    i += 1
print(toks)
print(re.findall(r'\d+|[-+*/()]', s))
```
</details>

### Drill 5: Sliding window max with deque
For `nums=[1,3,-1,-3,5,3,6,7]`, `k=3`, print the max of every window of size k (LC 239). Hint: keep indices in a deque with decreasing values.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
from collections import deque
nums, k = [1,3,-1,-3,5,3,6,7], 3
dq, out = deque(), []
for i, x in enumerate(nums):
    while dq and nums[dq[-1]] <= x:
        dq.pop()
    dq.append(i)
    if dq[0] <= i - k:
        dq.popleft()
    if i >= k - 1:
        out.append(nums[dq[0]])
print(out)  # [3, 3, 5, 5, 6, 7]
```
</details>

### Drill 6: Generator
Write a generator `chunks(lst, n)` that yields consecutive slices of size n. `list(chunks(list(range(7)), 3))` → `[[0,1,2],[3,4,5],[6]]`.

In [ ]:
# your attempt here

<details><summary><b>Answer</b></summary>

```python
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]
print(list(chunks(list(range(7)), 3)))
```
</details>

---
**Next:** `02_leetcode_practice.ipynb`. If you got through Parts 1–6 comfortably, you're ready.
Keep this notebook open as a reference while coding: looking up syntax is fine during practice, and by Oct 9 you won't need to.